# F1 Fantasy 2025: Round-by-Round Optimal Teams

This notebook generates the expected points (EV) and the mathematically optimal team configuration for every round of the 2025 season using the XGBoost model and Knapsack optimization.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
import os

# Add the root directory to sys.path to import from src
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.model import F1FantasyPredictor
from src.optimize_lp import solve_knapsack
import src.predict_ev as ev_engine

In [2]:
def get_optimal_team_summary(year, round_num, budget=100.0):
    """
    Generates EV, solves optimization, and returns a formatted summary DataFrame.
    """
    # 1. Generate EV Report
    ev_df = ev_engine.generate_ev_report(year, round_num)
    if ev_df is None:
        return None
    
    ev_file = f"../data/ev_reports/ev_report_{year}_R{round_num}.csv"
    ev_df.to_csv(ev_file, index=False)
    
    # 2. Solve Knapsack
    result = solve_knapsack(ev_file, budget)
    
    if not result:
        return pd.DataFrame({"Error": ["No valid team found"]})
    
    # 3. Format as requested (Rows 1-7 selection, 8 Cost, 9 Points)
    summary_data = []
    for i in range(5):
        summary_data.append({"Item": f"Driver {i+1}", "Selection": result["drivers"][i]})
    for i in range(2):
        summary_data.append({"Item": f"Constructor {i+1}", "Selection": result["constructors"][i]})
    
    summary_df = pd.DataFrame(summary_data)
    
    # Add Cost and Predicted Points
    extra_rows = pd.DataFrame([
        {"Item": "Total Cost", "Selection": f"${result['total_cost']:.1f}M"},
        {"Item": "Predicted Points", "Selection": f"{result['predicted_points']:.2f}"}
    ])
    
    summary_df = pd.concat([summary_df, extra_rows], ignore_index=True)
    return summary_df


## 2025 Season Analysis

Iterating through all available rounds in the 2025 dataset.

In [3]:
df_main = pd.read_csv("../data/processed_fantasy.csv")
rounds_2025 = sorted(df_main[df_main["year"] == 2025]["round"].unique())

round_summaries = {}

for r in rounds_2025:
    print(f"Analyzing Round {r}...")
    summary = get_optimal_team_summary(2025, r, budget=100.0)
    if summary is not None:
        round_summaries[r] = summary

print("Generation Complete.")

Analyzing Round 1...
Model trained with XGBoost. Features: 19. Test RMSE: 16.06
EV Report saved to /Users/joe/Documents/work/f1_fantasy_analysis/data/ev_reports/ev_report_2025_R1.csv

--- Top Drivers (Expected Points) ---
             asset_id  predicted_points  cost
      charles-leclerc         43.550297  25.9
         lando-norris         43.350662  29.0
       max-verstappen         36.637142  28.4
       lewis-hamilton         23.465366  24.2
       george-russell         23.423965  21.0
        oscar-piastri         21.543219  23.3
      carlos-sainz-jr         18.694828  13.1
          liam-lawson         16.542246  18.0
andrea-kimi-antonelli         14.001778  18.4
         pierre-gasly         12.385553  11.8
         isack-hadjar          9.738542   6.2
      fernando-alonso          8.009820   8.8
     franco-colapinto          7.995697   NaN
      alexander-albon          7.625653  12.0
         esteban-ocon          6.540954   7.3
       oliver-bearman          5.900412   

## Summary Tables

Below are the optimal configurations for each round.

In [7]:
from IPython.display import display, HTML

for r, df in round_summaries.items():
    display(HTML(f"<h3>Round {r}</h3>"))
    display(df)

,Item,Selection
0,Driver 1,charles-leclerc
1,Driver 2,carlos-sainz-jr
2,Driver 3,pierre-gasly
3,Driver 4,isack-hadjar
4,Driver 5,esteban-ocon
5,Constructor 1,ferrari
6,Constructor 2,aston-martin
7,Total Cost,$99.9M
8,Predicted Points,168.29


,Item,Selection
0,Driver 1,george-russell
1,Driver 2,lance-stroll
2,Driver 3,nico-hulkenberg
3,Driver 4,esteban-ocon
4,Driver 5,oliver-bearman
5,Constructor 1,red-bull
6,Constructor 2,mercedes
7,Total Cost,$98.5M
8,Predicted Points,135.68


,Item,Selection
0,Driver 1,carlos-sainz-jr
1,Driver 2,lewis-hamilton
2,Driver 3,nico-hulkenberg
3,Driver 4,fernando-alonso
4,Driver 5,esteban-ocon
5,Constructor 1,Mclaren
6,Constructor 2,haas
7,Total Cost,$97.6M
8,Predicted Points,156.32


,Item,Selection
0,Driver 1,esteban-ocon
1,Driver 2,liam-lawson
2,Driver 3,andrea-kimi-antonelli
3,Driver 4,pierre-gasly
4,Driver 5,isack-hadjar
5,Constructor 1,mercedes
6,Constructor 2,red-bull
7,Total Cost,$99.9M
8,Predicted Points,184.06


,Item,Selection
0,Driver 1,max-verstappen
1,Driver 2,oliver-bearman
2,Driver 3,alexander-albon
3,Driver 4,pierre-gasly
4,Driver 5,fernando-alonso
5,Constructor 1,mercedes
6,Constructor 2,haas
7,Total Cost,$98.6M
8,Predicted Points,131.22


...

,Item,Selection
0,Driver 1,max-verstappen
1,Driver 2,esteban-ocon
2,Driver 3,pierre-gasly
3,Driver 4,franco-colapinto
4,Driver 5,fernando-alonso
5,Constructor 1,red-bull
6,Constructor 2,williams
7,Total Cost,$99.7M
8,Predicted Points,142.27


,Item,Selection
0,Driver 1,george-russell
1,Driver 2,oliver-bearman
2,Driver 3,yuki-tsunoda
3,Driver 4,isack-hadjar
4,Driver 5,fernando-alonso
5,Constructor 1,mercedes
6,Constructor 2,williams
7,Total Cost,$100.0M
8,Predicted Points,120.99


,Item,Selection
0,Driver 1,max-verstappen
1,Driver 2,oliver-bearman
2,Driver 3,franco-colapinto
3,Driver 4,pierre-gasly
4,Driver 5,fernando-alonso
5,Constructor 1,mercedes
6,Constructor 2,williams
7,Total Cost,$99.8M
8,Predicted Points,163.21


,Item,Selection
0,Driver 1,lando-norris
1,Driver 2,esteban-ocon
2,Driver 3,nico-hulkenberg
3,Driver 4,fernando-alonso
4,Driver 5,franco-colapinto
5,Constructor 1,mercedes
6,Constructor 2,williams
7,Total Cost,$99.6M
8,Predicted Points,151.77


,Item,Selection
0,Driver 1,max-verstappen
1,Driver 2,carlos-sainz-jr
2,Driver 3,franco-colapinto
3,Driver 4,fernando-alonso
4,Driver 5,nico-hulkenberg
5,Constructor 1,mercedes
6,Constructor 2,williams
7,Total Cost,$99.9M
8,Predicted Points,149.88


In [8]:
## Top 5 Teams for Round 1
ev_file_r1 = "../data/ev_reports/ev_report_2025_R1.csv"
top_5_teams = solve_knapsack(ev_file_r1, budget=100.0, top_n=5)

top_5_summaries = []
for idx, team in enumerate(top_5_teams):
    summary_data = []
    for i in range(5):
        summary_data.append({"Item": f"Driver {i+1}", "Selection": team["drivers"][i]})
    for i in range(2):
        summary_data.append({"Item": f"Constructor {i+1}", "Selection": team["constructors"][i]})
    
    summary_df = pd.DataFrame(summary_data)
    extra_rows = pd.DataFrame([
        {"Item": "Total Cost", "Selection": f"${team['total_cost']:.1f}M"},
        {"Item": "Predicted Points", "Selection": f"{team['predicted_points']:.2f}"}
    ])
    summary_df = pd.concat([summary_df, extra_rows], ignore_index=True)
    top_5_summaries.append(summary_df)

print("Top 5 Optimal Teams for 2025 Round 1:")
for i, df in enumerate(top_5_summaries):
    display(HTML(f"<h4>Rank {i+1}</h4>"))
    display(df)

Iterating through driver combinations (19 active drivers)...
Top 5 Optimal Teams for 2025 Round 1:


,Item,Selection
0,Driver 1,charles-leclerc
1,Driver 2,carlos-sainz-jr
2,Driver 3,pierre-gasly
3,Driver 4,isack-hadjar
4,Driver 5,esteban-ocon
5,Constructor 1,ferrari
6,Constructor 2,aston-martin
7,Total Cost,$99.9M
8,Predicted Points,168.29


,Item,Selection
0,Driver 1,charles-leclerc
1,Driver 2,carlos-sainz-jr
2,Driver 3,pierre-gasly
3,Driver 4,isack-hadjar
4,Driver 5,oliver-bearman
5,Constructor 1,ferrari
6,Constructor 2,aston-martin
7,Total Cost,$99.3M
8,Predicted Points,167.65


,Item,Selection
0,Driver 1,carlos-sainz-jr
1,Driver 2,pierre-gasly
2,Driver 3,isack-hadjar
3,Driver 4,fernando-alonso
4,Driver 5,esteban-ocon
5,Constructor 1,ferrari
6,Constructor 2,red-bull
7,Total Cost,$99.5M
8,Predicted Points,167.15


,Item,Selection
0,Driver 1,carlos-sainz-jr
1,Driver 2,pierre-gasly
2,Driver 3,isack-hadjar
3,Driver 4,fernando-alonso
4,Driver 5,oliver-bearman
5,Constructor 1,ferrari
6,Constructor 2,red-bull
7,Total Cost,$98.9M
8,Predicted Points,166.51


,Item,Selection
0,Driver 1,charles-leclerc
1,Driver 2,carlos-sainz-jr
2,Driver 3,pierre-gasly
3,Driver 4,isack-hadjar
4,Driver 5,fernando-alonso
5,Constructor 1,ferrari
6,Constructor 2,haas
7,Total Cost,$99.9M
8,Predicted Points,165.60
